# Step 4 — the VCD gate: does the prior assert `Clip` harder without pixels?

**August plan, week 1.** Spec: `local/tasks/roadmap-percepcion-rl.md` B.3. Not a rung — the
ladder is closed. Trains nothing; two forward passes per frame.

## Pre-registration — declared BEFORE the run

On frames where `Clip` is a **false positive**, compare the **softmax mass share** of `Clip`
(never the raw logit — what picks the token is the ranking) with the real frame vs a degraded one:

| result | reading | verdict |
|---|---|---|
| p(Clip) **sinks** | the model does use the image; the error is elsewhere | 🔴 **dies** |
| p(Clip) **unchanged** (abs delta < eps) | `(1+a)L - aL = L` so VCD is an exact **no-op** | 🔴 **dies** |
| p(Clip) **rises** | the prior asserts `Clip` harder without pixels, so subtracting it moves mass away | 🟢 **build step 8** |

**Declared parameters:** `SIGMA = 25.0` (Gaussian pixel noise, 0-255 units), `EPS = 0.02`
(the band called *unchanged*). Neither is tuned — sweeping sigma until a verdict appears is the
multiplicity problem in a lab coat.

## The blocking control

*"Unchanged"* is also what a corruption **too weak to affect the model** looks like — a measurement
artifact that would read as a legitimate kill. `manipulation_check` requires evidence that the
degradation moved the output distribution at all. **If it fails, this notebook has no verdict.**
Same defect Rodrigo's negative-control amendment fixes in step 6.

**Decides:** whether step 8 exists. VCD touches the container's answer path, so before Sep 1 or never.

In [ ]:
# --- parameters (papermill) -----------------------------------------------------
# Comments ABOVE the assignment, never beside it: papermill silently skips a line it
# cannot parse and the -p is then ignored (measured on step 5, 2026-08-05).

# scored run whose predictions locate the Clip false positives
SRC_RUN = "/workspace/repo/experiments/21-recipe-sweep/runs/21_lr_2e4_v1/ep3_full"
RUN = "21_lr_2e4_v1"
CKPT_NAME = "checkpoint-2703"
# Gaussian pixel noise, 0-255 units. PRE-DECLARED, not tuned.
SIGMA = 25.0
# band within which real and degraded are called UNCHANGED. PRE-DECLARED.
EPS = 0.02
# blocking: the corruption must move entropy or top-prob by at least this
MIN_SHIFT = 0.01
SEED = 42
# True -> 24 frames. Flip to False only AFTER the smoke reads.
SMOKE = True
DATA_ROOT = "/workspace/orena-data"

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
import sys, time, json, gc
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd()
while not (REPO / "src" / "frame").is_dir():
    assert REPO != REPO.parent, "run me from inside the repo"
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "experiments" / "28-vcd-gate" / "_models"))

EXP = REPO / "experiments" / "28-vcd-gate"
TAG = "smoke" if SMOKE else "full"
OUT = EXP / "runs" / f"step4_vcd_{RUN}" / TAG
OUT.mkdir(parents=True, exist_ok=True)

DATA_ROOT = next((d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
                  if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"
print(f"OK    data_root {DATA_ROOT}")
print(f"OK    out       {OUT}")
print(f"      sigma={SIGMA} eps={EPS} min_shift={MIN_SHIFT} smoke={SMOKE}")

In [ ]:
# --- locate the frames where `Clip` is a FALSE POSITIVE -------------------------
from frame import metrics
from frame.metrics import read_fo_class

src = Path(SRC_RUN)
assert src.is_dir(), f"scored run not found: {src}"
res = pd.read_csv(src / "results.csv")
assert "answer_format" in res.columns, "results.csv has no answer_format column"
preds = json.loads((src / "predictions.json").read_text())
refs = json.loads((src / "references.json").read_text())
pred_by = {p["qID"]: p.get("content", "") for p in preds}
gold_by = {r["qID"]: r.get("answer", "") for r in refs}

valid_lower = {n.lower(): n for n in metrics._load_fotype().names()}
fo = res[res.answer_format == "fo_class"]
rows = []
for q in fo.qID:
    g = read_fo_class(gold_by.get(q, ""), valid_lower)
    p = read_fo_class(pred_by.get(q, ""), valid_lower)
    if p and "Clip" in p and (g is None or "Clip" not in g):
        rows.append(q)
fp = pd.DataFrame({"qID": rows})
print(f"      fo_class questions: {len(fo)}")
print(f"      Clip FALSE POSITIVES: {len(fp)}  ({len(fp)/max(len(fo),1):.1%})")
assert len(fp) >= 10, f"only {len(fp)} Clip FPs — too few to read a gate on"
if SMOKE:
    fp = fp.sample(n=min(24, len(fp)), random_state=SEED)
print(f"      using {len(fp)} frames")

In [ ]:
# --- the checkpoint, resolved by LOOKING across the pod's four checkouts --------
CAND = [Path(p) / "experiments/21-recipe-sweep/runs" / RUN / "merged" / CKPT_NAME
        for p in ("/workspace/repo", "/workspace/repo_leo", "/workspace/repo_rodri",
                  "/workspace/repo_yyy", str(REPO))]
MERGED = next((d for d in CAND if d.is_dir() and any(d.iterdir())), None)
assert MERGED is not None, "merged checkpoint not found:\n  " + "\n  ".join(map(str, CAND))
print(f"OK    merged {MERGED}")

In [ ]:
# --- two forward passes per frame: real and degraded ---------------------------
import vcd
from frame.config import BaselineConfig
from frame.data import FrameProvider, load_frame_items
from frame.engine import QwenFrameEngine

cfg = BaselineConfig(data_root=DATA_ROOT, model_path=MERGED, out_dir=OUT,
                     run_name=TAG, max_pixels=1280*720, seed=SEED)
items = {it.request.qID: it for it in load_frame_items(cfg, splits=("test",))}
sel = [items[q] for q in fp.qID if q in items]
assert len(sel) == len(fp), f"{len(fp) - len(sel)} qIDs have no frame item"

CLASSES = list(metrics._load_fotype().names())
prov = FrameProvider(cfg)
eng = QwenFrameEngine(cfg)
eng.load()
assert eng.model is not None and eng.processor is not None, "engine.load() did not load"

t0 = time.perf_counter()
pairs = []
for it in sel:
    prov.ensure_reader(it)
    img = prov.get_frame(it)
    r = vcd.class_mass(eng, img, it.request.question, CLASSES)
    d = vcd.class_mass(eng, vcd.degrade(img, SIGMA, SEED), it.request.question, CLASSES)
    pairs.append((r, d))
eng.unload()
del eng
gc.collect()
print(f"      {len(pairs)} frame pairs in {time.perf_counter() - t0:.0f}s")

In [ ]:
# --- BLOCKING: did the corruption move the model at all? ------------------------
mc = vcd.manipulation_check(pairs, min_shift=MIN_SHIFT)
print(json.dumps(mc, indent=2))
assert mc["passes"], (
    f"GATE — the degradation moved the output distribution by only {mc['moved']:.4f} "
    f"(< {MIN_SHIFT}). 'Unchanged' would be a measurement artifact, not a verdict. "
    "Raise SIGMA and re-run; do NOT read a result from this.")
print("\nOK    the corruption demonstrably perturbs the model — a null is now readable")

In [ ]:
# --- the pre-registered three-way verdict --------------------------------------
real = np.array([r["Clip"] for r, _ in pairs])
degr = np.array([d["Clip"] for _, d in pairs])
per_q = pd.DataFrame({"qID": [it.request.qID for it in sel],
                      "p_clip_real": real, "p_clip_degraded": degr,
                      "delta": degr - real})
per_q.to_csv(OUT / "per_question.csv", index=False)

v = vcd.verdict(real, degr, eps=EPS)
v["manipulation_check"] = mc
(OUT / "verdict.json").write_text(json.dumps(v, indent=2))
print(json.dumps(v, indent=2))
mark = "GREEN" if v["verdict"] == "BUILD" else "RED"
print(f"\n[{mark}] {v['verdict']} — {v['reading']}")
print("   BUILD -> step 8 (VCD) exists and enters before Sep 1.")
print("   DIES  -> step 8 does not exist; nothing else in the plan changes.")

## Reading it

- **The verdict is the cell above, at the declared sigma and eps.** Any sweep is exploratory and
  does not decide — the decision was registered before the run.
- **Mass share, not logits.** A logit shift that lifts every candidate equally changes no ranking
  and therefore no answer.
- **`Clip` mass is summed over every first-token that could begin the name**, bare and
  space-prefixed, upper and lower case. A single token id would understate it and bias the gate
  toward "sinks".
- **If the manipulation check fails there is no verdict.** Raise sigma and re-run. A null under an
  ineffective corruption is not evidence of anything.
- **This gate decides only whether step 8 exists.** Nothing else in the August plan moves either way.